In [4]:
#!/usr/bin/env python3
"""
nas_portal_download.py

Recursive downloader for NASA NAS Data Portal directory listings (HTML index).

Example file naming:
  DYAMOND_c1440_llc2160.COLLECTION.YYYYMMDD_HHMMz.nc4

Features:
- Recursive crawl (like wget -r -np)
- Filter by collection(s)
- Filter by start/end UTC datetime
- Parallel downloads
- Retries + backoff
- Best-effort resume (HTTP Range) when supported
- Optionally skip "bad_files.txt" at the root

Requires:
  pip install requests beautifulsoup4 python-dateutil
  
  @hectorg (HectorTG200)
  
"""

import os
import re
import time
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse
from typing import Optional, List, Tuple, Set
import requests
from bs4 import BeautifulSoup
from dateutil.parser import isoparse
from concurrent.futures import ThreadPoolExecutor, as_completed

# ----------------------------
# Configuration helpers
# ----------------------------
TS_RE = re.compile(r"\.(\d{8})_(\d{4})z\.(nc4|nc)$", re.IGNORECASE)
UA = "Mozilla/5.0 (Jupyter NAS downloader)"

def safe_mkdir(p):
    os.makedirs(p, exist_ok=True)

def normalize_base(url):
    return url if url.endswith("/") else url + "/"

def within_same_tree(base, u):
    b = urlparse(base)
    c = urlparse(u)
    return (b.scheme, b.netloc) == (c.scheme, c.netloc) and c.path.startswith(b.path)

def parse_datetime(fname):
    m = TS_RE.search(fname)
    if not m:
        return None
    ymd, hm = m.group(1), m.group(2)
    return datetime(
        int(ymd[:4]), int(ymd[4:6]), int(ymd[6:8]),
        int(hm[:2]), int(hm[2:4]), tzinfo=timezone.utc
    )

# ----------------------------
# Crawl directory listing
# ----------------------------
def crawl_directory(base_url, session, recursive=True, timeout=60):
    base_url = normalize_base(base_url)
    visited = set()
    queue = [base_url]
    files = []

    while queue:
        page = queue.pop(0)
        if page in visited:
            continue
        visited.add(page)

        html = session.get(page, timeout=timeout).text
        soup = BeautifulSoup(html, "html.parser")

        for a in soup.find_all("a", href=True):
            u = urljoin(page, a["href"])
            if not within_same_tree(base_url, u):
                continue

            if u.endswith("/") and recursive:
                queue.append(u)
            elif u.lower().endswith((".nc4", ".nc")):
                files.append(u)

    return sorted(set(files))

# ----------------------------
# Download logic
# ----------------------------
def download_file(session, url, out_dir, resume=True, timeout=120):
    rel = urlparse(url).path.split("/holding/")[-1]
    out = os.path.join(out_dir, rel)
    tmp = out + ".part"
    safe_mkdir(os.path.dirname(out))

    if os.path.exists(out):
        return f"SKIP {out}"

    headers = {}
    mode = "wb"

    if resume and os.path.exists(tmp):
        start = os.path.getsize(tmp)
        headers["Range"] = f"bytes={start}-"
        mode = "ab"

    with session.get(url, stream=True, headers=headers, timeout=timeout) as r:
        r.raise_for_status()
        with open(tmp, mode) as f:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

    os.rename(tmp, out)
    return f"OK   {out}"

# ----------------------------
# Main notebook function
# ----------------------------
def download_geosgcm_surf(
    base_url,
    out_dir,
    start=None,
    end=None,
    workers=4,
    recursive=True,
):
    session = requests.Session()
    session.headers.update({"User-Agent": UA})

    files = crawl_directory(base_url, session, recursive=recursive)
    print(f"Found {len(files)} NetCDF files")

    # time filtering
    if start or end:
        start = isoparse(start).astimezone(timezone.utc) if start else None
        end   = isoparse(end).astimezone(timezone.utc) if end else None
        keep = []
        for u in files:
            dt = parse_datetime(os.path.basename(u))
            if dt is None:
                continue
            if start and dt < start:
                continue
            if end and dt > end:
                continue
            keep.append(u)
        files = keep
        print(f"After time filter: {len(files)} files")

    safe_mkdir(out_dir)

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = [ex.submit(download_file, session, u, out_dir) for u in files]
        for f in as_completed(futures):
            print(f.result())


## Download a data subset

In [5]:
collection = "geosgcm_surf"

BASE = "https://data.nas.nasa.gov/geosecco/c1440_llc2160/holding/"+collection+"/"
OUT  = "/u/bura-z2/hectorg/COAS/data/geos/"+collection+"/"  ## << ===== Collection

download_geosgcm_surf(
    base_url=BASE,
    out_dir=OUT,
    workers=4,            # keep small (portal is fragile)
    start="2020-02-01T00:00Z",  # optional
    end="2020-02-02T23:00Z",    # optional
)


Found 10365 NetCDF files
After time filter: 47 files
OK   /u/bura-z2/hectorg/COAS/data/geos/geosgcm_surf/geosgcm_surf/DYAMOND_c1440_llc2160.geosgcm_surf.20200201_0330z.nc4
OK   /u/bura-z2/hectorg/COAS/data/geos/geosgcm_surf/geosgcm_surf/DYAMOND_c1440_llc2160.geosgcm_surf.20200201_0130z.nc4
OK   /u/bura-z2/hectorg/COAS/data/geos/geosgcm_surf/geosgcm_surf/DYAMOND_c1440_llc2160.geosgcm_surf.20200201_0230z.nc4
OK   /u/bura-z2/hectorg/COAS/data/geos/geosgcm_surf/geosgcm_surf/DYAMOND_c1440_llc2160.geosgcm_surf.20200201_0030z.nc4
OK   /u/bura-z2/hectorg/COAS/data/geos/geosgcm_surf/geosgcm_surf/DYAMOND_c1440_llc2160.geosgcm_surf.20200201_0430z.nc4
OK   /u/bura-z2/hectorg/COAS/data/geos/geosgcm_surf/geosgcm_surf/DYAMOND_c1440_llc2160.geosgcm_surf.20200201_0530z.nc4
OK   /u/bura-z2/hectorg/COAS/data/geos/geosgcm_surf/geosgcm_surf/DYAMOND_c1440_llc2160.geosgcm_surf.20200201_0630z.nc4
OK   /u/bura-z2/hectorg/COAS/data/geos/geosgcm_surf/geosgcm_surf/DYAMOND_c1440_llc2160.geosgcm_surf.20200201_0730z